<a href="https://colab.research.google.com/github/DanielRegaladoUMiami/sql-agent-llmops/blob/main/training/notebooks/train_chart_reasoner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Chart Reasoner · Fine-tuning Notebook

Fine-tune **Phi-3 Mini 3.8B** to map `(natural-language question + SQL result schema) → storytelling chart spec` using QLoRA with Unsloth.

- **Dataset**: [`DanielRegaladoCardoso/chart-reasoning-mix-v1`](https://huggingface.co/datasets/DanielRegaladoCardoso/chart-reasoning-mix-v1) — ~75 k rows (nvBench real + OpenAI storytelling synth)
- **Base model**: [`unsloth/Phi-3-mini-4k-instruct-bnb-4bit`](https://huggingface.co/unsloth/Phi-3-mini-4k-instruct-bnb-4bit)
- **Hardware**: Colab **T4 free tier** ✅
- **Expected time**: ~2–3 h for 1 epoch

> [`https://github.com/DanielRegaladoUMiami/sql-agent-llmops`](https://github.com/DanielRegaladoUMiami/sql-agent-llmops)

### Storytelling principles baked into the output schema

Synth rows include full chart spec: `chart_type`, `encoding`, **insight-driven title**, `subtitle`, `annotations`, `sort`, `color_strategy`, `highlight_value`, `axis_format`, `rationale` — distilled from Tufte / Knaflic / Few.


## 1 · Check GPU

In [ ]:
!nvidia-smi

## 2 · Install

In [ ]:
!pip install -q "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets xformers


## 3 · HuggingFace login

In [ ]:
from huggingface_hub import login
login()


## 4 · Load Phi-3 Mini in 4-bit

Unsloth's Phi-3 support has some version churn — this cell tries the
pre-quantized Unsloth repo first and falls back to Microsoft's official
Phi-3-mini-4k-instruct (with bitsandbytes 4-bit on the fly) if needed.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048

PRIMARY  = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
FALLBACK = "microsoft/Phi-3-mini-4k-instruct"

def load_with_fallback(primary, fallback, max_seq_length=MAX_SEQ_LEN):
    try:
        print(f"Trying primary: {primary}")
        return FastLanguageModel.from_pretrained(
            model_name=primary, max_seq_length=max_seq_length,
            load_in_4bit=True, dtype=None,
        )
    except Exception as e:
        print(f"⚠️  Primary failed ({type(e).__name__}). Falling back to {fallback}.")
        return FastLanguageModel.from_pretrained(
            model_name=fallback, max_seq_length=max_seq_length,
            load_in_4bit=True, dtype=None,
        )

model, tokenizer = load_with_fallback(PRIMARY, FALLBACK)


## 5 · Attach LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                  = 16,
    target_modules     = ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"],
    lora_alpha         = 32,
    lora_dropout       = 0,
    bias               = "none",
    use_gradient_checkpointing = "unsloth",
    random_state       = 42,
)


## 6 · Load dataset

In [ ]:
from datasets import load_dataset
import json

ds = load_dataset("DanielRegaladoCardoso/chart-reasoning-mix-v1")
print(ds)
print()
print("Sample instruction:", ds["train"][0]["instruction"])
print("Sample chart_spec keys:", list(ds["train"][0]["chart_spec"].keys())
      if isinstance(ds["train"][0]["chart_spec"], dict) else "(stored as JSON string)")


## 7 · Format as chat messages

We teach the model to output the **full storytelling JSON spec** — not just
`chart_type` — so the resulting model produces insight-driven titles,
annotations, sorting rules, and rationales at inference time.

In [ ]:
SYSTEM_PROMPT = (
    "You are an elite data-visualization expert. Given a natural-language "
    "question and the schema of a SQL result set, produce the ideal chart "
    "specification as JSON with: chart_type, encoding (x/y/color/size/facet), "
    "insight-driven title, subtitle, annotations, sort, color_strategy, "
    "highlight_value, axis_format, rationale. Respond with JSON only."
)

def _parse(v):
    """Parse a JSON-string field (how this dataset stores nested dicts)."""
    if isinstance(v, str):
        try: return json.loads(v)
        except: return {}
    return v or {}

def to_chat(row):
    profile = _parse(row["data_profile"])
    cols = profile.get("columns", [])
    user_content = (
        f"Question: {row['instruction']}\n\n"
        f"SQL result columns: {json.dumps(cols)}"
    )
    spec = row["chart_spec"] if isinstance(row["chart_spec"], str) else json.dumps(row["chart_spec"])
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": spec},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

train_ds = ds["train"].map(to_chat, remove_columns=ds["train"].column_names)
eval_ds  = ds["validation"].map(to_chat, remove_columns=ds["validation"].column_names)
print(train_ds[0]["text"][:1000])


## 7.5 · Filter rows that exceed MAX_SEQ_LEN

In [ ]:
def fits(row):
    return len(tokenizer.encode(row["text"], add_special_tokens=False)) <= MAX_SEQ_LEN

_bt, _be = len(train_ds), len(eval_ds)
train_ds = train_ds.filter(fits, num_proc=2)
eval_ds  = eval_ds.filter(fits,  num_proc=2)
print(f"Train: kept {len(train_ds):,} / {_bt:,} ({100*len(train_ds)/max(1,_bt):.1f}%)")
print(f"Eval : kept {len(eval_ds):,} / {_be:,} ({100*len(eval_ds)/max(1,_be):.1f}%)")


## 8 · Trainer config

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = "chart_reasoner_adapter",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio                = 0.03,
    num_train_epochs            = 1,
    learning_rate               = 2e-4,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 25,
    save_steps                  = 500,
    save_total_limit            = 2,
    optim                       = "adamw_8bit",
    lr_scheduler_type           = "cosine",
    seed                        = 42,
    report_to                   = "none",
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LEN,
    packing                     = False,
)

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_ds,
    eval_dataset   = eval_ds.select(range(min(500, len(eval_ds)))),
    args           = training_args,
)


## 9 · Train

In [ ]:
trainer.train()

## 10 · Inference test

In [ ]:
FastLanguageModel.for_inference(model)

sample = ds["test"][0]
profile_s = _parse(sample["data_profile"])
cols = profile_s.get("columns", [])
user_content = f"Question: {sample['instruction']}\n\nSQL result columns: {json.dumps(cols)}"
print("Gold chart_spec:", _parse(sample["chart_spec"]))
print()

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_content},
]
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids, max_new_tokens=600, do_sample=False, temperature=0)
print(tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True))


## 11 · Push adapter to HuggingFace

In [ ]:
model.push_to_hub_merged(
    "DanielRegaladoCardoso/chart-reasoner-phi3-mini-lora",
    tokenizer,
    save_method="lora",
    token="**",
)


In [ ]:
model.push_to_hub(
    "DanielRegaladoCardoso/chart-reasoner-phi3-mini-adapter-only",
    token="**",
)
tokenizer.push_to_hub(
    "DanielRegaladoCardoso/chart-reasoner-phi3-mini-adapter-only",
    token="**",
)

## ✅ Done

Your adapter is at:
**https://huggingface.co/DanielRegaladoCardoso/chart-reasoner-phi3-mini-lora**
